# Test python verification packages with synthetic dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#pip install bomwater

In [3]:
#help(gg.gauge_getter)

In [4]:
#!pip install -e /datasets/work/lw-hydrofct/work/common/Software/xarray_utilities/

In [5]:
import sys,glob
print("Python version")
print (sys.version)
#print("Version info.")
#print (sys.version_info)

Python version
3.10.11 | packaged by conda-forge | (main, May 10 2023, 18:58:44) [GCC 11.3.0]


### Import Various Necessary Libraries Dask and Related Libraries

In [6]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask
import os
import uuid

In [7]:
os.environ['USE_PYGEOS'] = '0'
import random
import datetime as dt
import pytz
#from decimal import *#
import geopandas as gpd 
import fiona 
import numpy as np
import math
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import xarray as xr
from PIL import Image
import mdba_gauge_getter as gg
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns

In [8]:
#!pip install seaborn --upgrade

### Add proxy binaries to path

In [9]:
pwd = !echo ${PWD}

In [10]:
pwd

['/datasets/work/d61-coastal-forecasting-wp3/work/shr015/Code/python_singularity/swift_notebooks/fcst_verf/Hawkesbury']

### Extend the local python paths with some network drives

In [11]:
extension_python_paths = [ os.environ["HOME"] + '/lib/python3.10/site-packages']
[sys.path.append(an_ext) for an_ext in extension_python_paths]

[None]

### Specify a python exe used by SLURM to create the dask workers


In [12]:
containered_python_exe = f"srun --export=ALL -n $SLURM_NTASKS -c $SLURM_CPUS_PER_TASK   singularity run {os.environ['SINGULARITY_CONTAINER']} python"

### If you run this cell after creating a cluster it will close that cluster 

In [13]:
try:
    cluster.close()
except:
    pass

### Create a cluster
env_extra sets the worker specific environment parameters   

PYTHONPATH is set to include the extension ptyhon paths and the SINGULARITY_BINDPATH bind paths of this jupyter environment for passing to the workers

In [14]:
job_suffix = os.environ['JOB_SUFFIX'] if 'JOB_SUFFIX' in os.environ.keys() else str(uuid.uuid4())[:8]

In [15]:
job_suffix

'b361d877'

## List available project codes

In [16]:
!get_project_codes

/bin/bash: line 1: get_project_codes: command not found


In [17]:
defined = 'NC_IN_GLOB' in os.environ.keys()
if not defined:
    print("WARNING, project code note defined defaulting")
    project_code = 'OD-230112'
else:
    project_code = os.environ['NC_IN_GLOB']

WARNING, project code note defined defaulting


In [18]:
project_code

'OD-230112'

In [19]:
job_extra = f'--account {project_code}'

In [20]:
process_number = 2

In [21]:
cluster = SLURMCluster(
    cores=2, memory="12G", processes=process_number,
    walltime="02:59:00",
    interface='ib0',
    death_timeout=480,
    job_name = f'dask-worker-{job_suffix}',
    job_extra_directives = [job_extra],
    job_script_prologue=[
              'module load singularity', # ensure singularity is loaded
              'export PYTHONPATH=' + ':'.join(extension_python_paths),
              'export SINGULARITY_BINDPATH=' + os.environ['SINGULARITY_BIND'], 
              'export SINGULARITYENV_PREPEND_PATH='+ str(pwd[0]) +':' + str(pwd[0]) + ',/srv/conda/envs/notebook/bin:/srv/conda/condabin:/srv/conda/bin'],
    worker_extra_args = [f'--local-directory={os.environ["SCRATCH3DIR"]}'],
    log_directory = '/home/shr015/singularity_slurm_log_directory',
    python=containered_python_exe,  # use pyhton in container
)

# Debug by running cluster.job_script()

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: tcp://10.150.202.155:32833
INFO:distributed.scheduler:  dashboard at:  http://10.150.202.155:8787/status


In [22]:
#cluster.job_script()

### Create a client, this will inject dask into xarray and the distributed cluster into dask

In [23]:
client = Client(cluster, timeout=240)
display(client)

INFO:distributed.scheduler:Receive client connection: Client-5feb976d-69c6-11ef-929c-70b5e8f03b1a
INFO:distributed.core:Starting established connection to tcp://10.150.202.155:34144


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.150.202.155:8787/status,
Dashboard: http://10.150.202.155:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.150.202.155:32833,Workers: 0
Dashboard: http://10.150.202.155:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [24]:
port = client.dashboard_link.split('/')[-2].split(':')[-1]
print(f"Try http://localhost:8888/proxy/{port}/status for the dask dashboard")

Try http://localhost:8888/proxy/8787/status for the dask dashboard


### Scale your workers

With the default config in SLURMCluster above each job will get create 2 workers, one per process and each will have 30gb ram and 4 cores

In [25]:
max_workers = 5

import time
for i in range(0, max_workers): 
    cluster.scale(jobs=i) #yes this looks weird requesting n+1 workers everytime but really it only requests 1 new worker each time
    time.sleep(5)

timeout = 600   # seconds till timeout, timeout if cluster not up in 10 minutes
timeout_start = time.time()
while len(client.ncores().keys())*process_number < max_workers -1:
    if (time.time() > timeout_start+timeout):
        raise Exception(f"Failed to start enough workers in {timeout} seconds, {len(cluster.workers)} started")
    time.sleep(1)

INFO:distributed.scheduler:Register worker <WorkerState 'tcp://10.150.202.44:47019', name: SLURMCluster-1-0, status: init, memory: 0, processing: 0>
INFO:distributed.scheduler:Starting worker compute stream, tcp://10.150.202.44:47019
INFO:distributed.core:Starting established connection to tcp://10.150.202.44:51142
INFO:distributed.scheduler:Register worker <WorkerState 'tcp://10.150.202.44:32963', name: SLURMCluster-0-0, status: init, memory: 0, processing: 0>
INFO:distributed.scheduler:Starting worker compute stream, tcp://10.150.202.44:32963
INFO:distributed.core:Starting established connection to tcp://10.150.202.44:51150
INFO:distributed.scheduler:Register worker <WorkerState 'tcp://10.150.202.44:41993', name: SLURMCluster-1-1, status: init, memory: 0, processing: 0>
INFO:distributed.scheduler:Starting worker compute stream, tcp://10.150.202.44:41993
INFO:distributed.core:Starting established connection to tcp://10.150.202.44:51136
INFO:distributed.scheduler:Register worker <Worke

In [26]:
len(client.ncores())

8

In [27]:
#%load_ext line_profiler

# Petrichor setting
TO run this notebook in petrichor
 - Uncomment out following until "EASI-hub setting"
 - Comment out from "EASI-hub setting" until "End of setting"

In [28]:
#sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/gcm-analysis/scripts')
sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/python_functions/')
import common_functions as cf,plot_utils
from netcdf_utility import nc_utils  #'/datasets/work/lw-hydrofct/work/common/Software/python_functions/netcdf_utility
sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/python_verification/')
import vrf_scores
import crps_class

In [29]:
client.upload_file('/datasets/work/lw-hydrofct/work/common/Software/python_functions/netcdf_utility/nc_utils.py');
client.upload_file('/datasets/work/lw-hydrofct/work/common/Software/python_functions/common_functions.py');
client.upload_file('/datasets/work/lw-hydrofct/work/common/Software/python_verification/vrf_scores.py');

In [30]:
#sys.path.append('/datasets/work/lw-hydrofct/work/common/Software/bjp-g/src')
#import pybjp

In [31]:
#sys.path.append('/datasets/work/lw-hydrofct/work/common/Projects/DPE_Headroom/Code/Matlab/demandsFcts/2Code/f2_waterDemandsBjp')
#import bjpmodel

## Installing libraries
Installing libraries will install to singularity/.local inside the singularity environment this appears at $HOME/.local.

It isn't possible to directly install libraries on hpc nodes apart from interactive and login and in the future it may not be possible to install libraries even on these nodes. To bypass this we use pip download to download libraries and dependencies as whl files to a network drive accessible to HPC machines.

For example we might want to install the library nltk for natural language processing. We have a shared path at /datasets/work/oa-sle/work/python_envs/ben_nglp_1/ accessible to ppts-login. Via windows we can execute a script to download the packages. The cell below is an example script generator for windows downloads. The variable values can be changed to generate a script customized for particular packages.

In [32]:
package_to_install = ''#'dea_tools'#'line_profiler' # insert package to install here e.g. "openpyxl"
my_library_path = "/datasets/work/lw-werp-cafs/work/shr015/Code/python_singularity_envs"
my_ident = "shr015"
container = "/datasets/work/lw-wa-bpa/work/WORKING/ENVS/singularity/pangeo-latest.sif"
windows_script = f"""ssh {my_ident}@petrichor-login.hpc.csiro.au "module load singularity && export SINGULARITY_BINDPATH={my_library_path}:{my_library_path} && export CONTAINER={container} && singularity run {container} /bin/bash -c 'cd {my_library_path}  && /srv/conda/envs/notebook/bin/pip download {package_to_install}'"""

In [33]:
if package_to_install != '':
    print(windows_script)

In [34]:
#%%sh
#pip3 install --find-links /datasets/work/lw-werp-cafs/work/shr015/Code/python_singularity_envs --no-index /datasets/work/lw-werp-cafs/work/shr015/Code/python_singularity_envs/dea_tools-0.2.7-py3-none-any.whl

# EASI-hub setting
TO run this notebook in EASI-hub
 - Comment out from "Petrichor setting" until "EASI-hub setting"
 - Uncomment out following until "End of setting"

In [35]:
#sys.path.extend(['/home/jovyan/nbic-workflow', '/home/jovyan/xarray_utilities','/home/jovyan/climate_change_bias_correction'])

In [36]:
# import xrutils
# import nbic_utils
# import utils
# import delta_correction as wc
# import s3fs

## Cluster

In [37]:
# client, cluster = nbic_utils.easi_cluster(number_of_workers=10)
# client

### End of setting

In [1]:
!pip install scoringrules

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.6 MB/s eta 0:00:00
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 65.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 MB 64.6 MB/s eta 0:00:0000:0100:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cmip6-preprocessing 0.6.0 requires pint-xarray, which is not installed.
apache-airflow 2.1.3 requires attrs<21.0,>=20.0, but you have

# Now add you usual codes from here

In [38]:
tstart = time.time()
print('Started: ', time.ctime(tstart))

Started:  Tue Sep  3 17:31:12 2024


In [39]:
dask.config.set(
    {"array.slicing.split_large_chunks": False } #True}
)  # to avoid creating the large chunk in the first place

In [172]:
def energy_score(forecasts, obs):

    # must have dimensions of (num_variables, num_ensembles). Variables can include different locations, time steps, climate variables etc

    num_samples = forecasts.shape[1]

    s1 = np.sqrt(np.sum(np.square(forecasts - obs[:, np.newaxis]), axis=0)).sum()

    pairwise_diffs = forecasts[:, :, np.newaxis] - forecasts[:, np.newaxis, :]
    s2 = np.sqrt(np.sum(np.square(pairwise_diffs), axis=0)).sum()

    es = (s1 / num_samples) - s2 / (2 * num_samples**2)
    
    return es

In [ ]:
import scoringrules
import xarray as xr
import numpy as np
from scores.probability import crps_for_ensemble
import properscoring
#np.random.seed(100)

In [238]:
def crps_from_empirical_cdf(pred, obs, dim=0):
    #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    n = pred.shape[dim]
    pred = np.sort(pred, axis=dim)
    ans = np.zeros_like(obs)

    # dx [F(x) - H(x-y)]^2 = dx [0 - 1]^2 = dx
    # val = ensemble[0] - truth
    val = (pred[0, :] - obs)
    #val = (pred[:, 0] - obs)
    ans += np.maximum(val, 0.0)

    for i in range(n - 1):
        x0 = pred[i, :]
        x1 = pred[i+1, :]

        cdf = (i + 1) / n

        # a. case y < x0
        val = (x1 - x0) * (cdf - 1) ** 2
        mask = obs < x0
        ans += val * mask

        # b. case x0 <= y <= x1
        val = (obs - x0) * cdf**2 + (x1 - obs) * (cdf - 1) ** 2
        mask = (obs >= x0) & (obs <= x1)
        ans += val * mask

        # c. case x1 < t
        mask = obs > x1
        val = (x1 - x0) * cdf**2
        ans += val * mask

    # dx [F(x) - H(x-y)]^2 = dx [1 - 0]^2 = dx
    val = obs - pred[-1, :]
    ans += np.maximum(val, 0.0)
    return ans

### Without NAN

In [257]:
#np.random.seed(100)
fcst_np = np.random.rand(20, 20, 20)
obs_np = np.random.rand(20, 20)

fcst = xr.DataArray(fcst_np, dims=["x", "y", "ens_mem"], coords={"x": np.arange(20), "y": np.arange(20), "ens_mem": np.arange(20)})
obs = xr.DataArray(obs_np, dims=["x", "y"], coords={"x": np.arange(20), "y": np.arange(20)})

scores_crps = crps_for_ensemble(fcst, obs, ensemble_member_dim="ens_mem")
print(f"scores {scores_crps.item()}")

ens_mem_dim = fcst.get_axis_num("ens_mem")

scoringrules_crps = scoringrules.crps_ensemble(obs, fcst, axis=ens_mem_dim, estimator="nrg")
print(f"scoringrules {scoringrules_crps.mean().item()}")

# since preserve_dims = None, reduce dimension to ens_mem for other packages
fcst_np2d = fcst_np.reshape(20*20,20)
obs_np1d = obs_np.reshape(20*20,)
vrf_scores_crps = vrf_scores.crps_ecdf_multiple(fcst_np2d,obs_np1d) 
print(f"vrf_scores_crps {vrf_scores_crps}")
properscoring_crps = np.mean(properscoring.crps_ensemble(obs_np1d,fcst_np2d))
print(f"properscoring.crps_ensemble {properscoring_crps}")
ecdf_crps = np.mean(crps_from_empirical_cdf(fcst_np2d.T,obs_np1d,dim=0))  #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
print(f"ecdf_crps {ecdf_crps}")          
assert ecdf_crps == scores_crps.item()

scores 0.17791152852407024
scoringrules 0.17791152852407024
vrf_scores_crps 0.1779115285240703
properscoring.crps_ensemble 0.1779115285240703
ecdf_crps 0.1779115285240703


AssertionError: 

In [287]:
#np.random.seed(100)
count=0
for i in range(100):
    fcst_np = np.random.rand(20, 20, 20)
    obs_np = np.random.rand(20, 20)
    
    fcst = xr.DataArray(fcst_np, dims=["x", "y", "ens_mem"], coords={"x": np.arange(20), "y": np.arange(20), "ens_mem": np.arange(20)})
    obs = xr.DataArray(obs_np, dims=["x", "y"], coords={"x": np.arange(20), "y": np.arange(20)})
    
    scores_crps = crps_for_ensemble(fcst, obs, ensemble_member_dim="ens_mem")
    #print(f"scores {scores_crps.item()}")
    
    ens_mem_dim = fcst.get_axis_num("ens_mem")
    
    scoringrules_crps = scoringrules.crps_ensemble(obs, fcst, axis=ens_mem_dim, estimator="nrg")
    #print(f"scoringrules {scoringrules_crps.mean().item()}")
    
    # since preserve_dims = None, reduce dimension to ens_mem for other packages
    fcst_np2d = fcst_np.reshape(20*20,20)
    obs_np1d = obs_np.reshape(20*20,)
    vrf_scores_crps = vrf_scores.crps_ecdf_multiple(fcst_np2d,obs_np1d) 
    #print(f"vrf_scores_crps {vrf_scores_crps}")
    properscoring_crps = np.mean(properscoring.crps_ensemble(obs_np1d,fcst_np2d))
    #print(f"properscoring.crps_ensemble {properscoring_crps}")
    ecdf_crps = np.mean(crps_from_empirical_cdf(fcst_np2d.T,obs_np1d,dim=0))  #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    #print(f"ecdf_crps {ecdf_crps}")          
    if ecdf_crps != scores_crps.item():
         count += 1
         print(f"scores1 {scores_crps.item()}")
         print(f"scores2 {ecdf_crps}")   
print(count)

scores1 0.18314846968875148
scores2 0.1831484696887515
scores1 0.17977316916580066
scores2 0.1797731691658007
scores1 0.1731948669979251
scores2 0.17319486699792513
scores1 0.16420104428278431
scores2 0.16420104428278434
scores1 0.1718357999706695
scores2 0.17183579997066947
scores1 0.17438597984637297
scores2 0.17438597984637302
scores1 0.18068321980406585
scores2 0.18068321980406588
scores1 0.17631097793821726
scores2 0.17631097793821732
scores1 0.17600564371159172
scores2 0.17600564371159177
scores1 0.17778859069354314
scores2 0.1777885906935431
scores1 0.17691911493448742
scores2 0.1769191149344874
scores1 0.1761150293236777
scores2 0.17611502932367773
scores1 0.1798554286109929
scores2 0.17985542861099282
scores1 0.17661215713941147
scores2 0.1766121571394115
scores1 0.17363786152871277
scores2 0.17363786152871274
scores1 0.17942452331954797
scores2 0.179424523319548
scores1 0.1778793608939254
scores2 0.17787936089392542
scores1 0.17953643189067114
scores2 0.1795364318906712
score

### With NAN

In [313]:
#np.random.seed(100)
fcst_np = np.random.rand(20, 20, 20)
obs_np = np.random.rand(20, 20)

In [340]:
fcst_np = np.random.rand(20, 20, 20)
obs_np = np.random.rand(20, 20)
count=0
for i in range(100):
    num_nan = 10
    # Generate random indices
    indices = np.random.choice(obs_np.size, num_nan, replace=False)
    # Convert 1D indices to 2D indices
    rows, cols = np.unravel_index(indices, obs_np.shape)
    # Assign np.nan to these indices
    obs_np[rows, cols] = np.nan
    fcst = xr.DataArray(fcst_np, dims=["x", "y", "ens_mem"], coords={"x": np.arange(20), "y": np.arange(20), "ens_mem": np.arange(20)})
    obs = xr.DataArray(obs_np, dims=["x", "y"], coords={"x": np.arange(20), "y": np.arange(20)})
    scores_crps = crps_for_ensemble(fcst, obs, ensemble_member_dim="ens_mem")
    #print(f"scores {scores_crps.item()}")
    ens_mem_dim = fcst.get_axis_num("ens_mem")
    # since preserve_dims = None, reduce dimension to ens_mem
    fcst_np2d = fcst_np.reshape(20*20,20)
    obs_np1d = obs_np.reshape(20*20,)
    # since other cannot handle nan, remove before applying
    mean_fcst = np.mean(fcst_np2d,axis=1) #
    nanindx = ~np.isnan(obs_np1d) & ~np.isnan(mean_fcst)
    obs_np1d = obs_np1d[nanindx]
    fcst_np2d = fcst_np2d[nanindx,:]  
    #scoringrules_crps = scoringrules.crps_ensemble(obs, fcst, axis=ens_mem_dim, estimator="nrg")
    #print(f"scoringrules {scoringrules_crps.mean().item()}")
    vrf_scores_crps = vrf_scores.crps_ecdf_multiple(fcst_np2d,obs_np1d) 
    #print(f"vrf_scores_crps {vrf_scores_crps}")
    properscoring_crps = np.mean(properscoring.crps_ensemble(obs_np1d,fcst_np2d))
    #print(f"properscoring.crps_ensemble {properscoring_crps}")
    ecdf_crps = np.mean(crps_from_empirical_cdf(fcst_np2d.T,obs_np1d,dim=0)) #https://docs.nvidia.com/deeplearning/modulus/modulus-core/_modules/modulus/metrics/general/crps.html
    #print(f"ecdf_crps {ecdf_crps}")
    #assert ecdf_crps == scores_crps.item()
    if ecdf_crps != scores_crps.item():
         count += 1
         #print(i)
         #print(f"scores1 {scores_crps.item()}")
         #print(f"scores2 {ecdf_crps}")   
print(count)

37
